# Spam Detection — Model Comparison

This notebook evaluates and compares **Logistic Regression**, **Random Forest**, and **XGBoost** on the SMS Spam Collection dataset.

We focus on metrics appropriate for **imbalanced classification** (87% ham, 13% spam) — in particular **PR-AUC** and **F1-score** — and explain why the winning model is the best choice.

> **Prerequisites:** run the pipeline before opening this notebook:
> ```bash
> python src/preprocess.py
> python src/train.py
> ```

## 1. Setup

In [ ]:
import os
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

# allow imports from src/
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(ROOT_DIR, "src"))

DATA_DIR = os.path.join(ROOT_DIR, "data")
MODELS_DIR = os.path.join(ROOT_DIR, "models")

MODEL_NAMES = ["LogisticRegression", "RandomForest", "XGBoost"]
COLORS = {
    "LogisticRegression": "#6366f1",
    "RandomForest": "#22c55e",
    "XGBoost": "#f59e0b",
}

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 2. Load test data and trained models

In [ ]:
X_test = sp.load_npz(os.path.join(DATA_DIR, "X_test.npz"))
y_test = joblib.load(os.path.join(DATA_DIR, "y_test.joblib"))

print("Test set: %d samples" % X_test.shape[0])
print("Class distribution: ham=%d, spam=%d (%.1f%% spam)" % (
    (y_test == 0).sum(),
    (y_test == 1).sum(),
    100 * y_test.mean(),
))

In [ ]:
models = {}
for name in MODEL_NAMES:
    path = os.path.join(MODELS_DIR, name + ".joblib")
    models[name] = joblib.load(path)
    print("Loaded: %s" % name)

## 3. Evaluate all models

We compute predictions and probabilities for each model, then calculate all relevant metrics.

In [ ]:
results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "y_pred": y_pred,
        "y_proba": y_proba,
    }

In [ ]:
# Build a summary DataFrame
summary = pd.DataFrame({
    name: {
        "Accuracy": m["accuracy"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1-Score": m["f1"],
        "ROC-AUC": m["roc_auc"],
        "PR-AUC": m["pr_auc"],
    }
    for name, m in results.items()
}).T

# Highlight the best value in each column
summary.style.highlight_max(axis=0, color="#bbf7d0").format("{:.4f}")

## 4. Why PR-AUC is the primary metric

Spam detection is a **class-imbalanced** problem (~87% ham, ~13% spam). This has direct consequences for which metrics we can trust:

**Accuracy is misleading.** A model that predicts "ham" for every single message still scores ~87% accuracy. It's useless as a spam filter, but accuracy says it's great.

**ROC-AUC is inflated.** ROC-AUC measures the trade-off between true positive rate and false positive rate. But when negatives massively outnumber positives, even a poor model rarely produces false positives *relative to the total negative count*. Look at the table above — all three models score between 0.96 and 0.99 on ROC-AUC, making it nearly impossible to differentiate them.

**PR-AUC focuses on the minority class.** PR-AUC (`average_precision_score`) measures how well the model maintains high precision as recall increases, without being propped up by the easy negative predictions. This is where meaningful model differences actually surface.

**F1-score** is also important — it captures the precision-recall trade-off at a single threshold. But PR-AUC evaluates performance across *all* thresholds, giving a more complete picture. We use both.

## 5. Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

for name, m in results.items():
    prec, rec, _ = precision_recall_curve(y_test, m["y_proba"])
    label = "%s  (PR-AUC = %.4f)" % (name, m["pr_auc"])
    ax.plot(rec, prec, label=label, color=COLORS[name], linewidth=2)

baseline = y_test.sum() / len(y_test)
ax.axhline(y=baseline, color="gray", linestyle="--", linewidth=1,
           label="Baseline (%.2f)" % baseline)

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curves", fontsize=15, fontweight="bold")
ax.legend(loc="lower left", fontsize=10)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

The PR curve shows how each model trades off precision for recall. Random Forest maintains the highest precision across nearly all recall levels, translating into the highest PR-AUC. XGBoost's curve drops earlier, reflecting its lower precision at moderate recall.

## 6. F1-Score Across Thresholds

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

best_thresholds = {}  # store for use in confusion matrices

for name, m in results.items():
    prec, rec, thresholds = precision_recall_curve(y_test, m["y_proba"])
    prec = prec[:-1]
    rec = rec[:-1]

    with np.errstate(divide="ignore", invalid="ignore"):
        f1_vals = np.where(
            (prec + rec) > 0,
            2 * prec * rec / (prec + rec),
            0.0,
        )

    best_idx = np.argmax(f1_vals)
    best_f1 = f1_vals[best_idx]
    best_thr = thresholds[best_idx]
    best_thresholds[name] = (best_thr, best_f1)

    label = "%s  (best F1 = %.4f @ %.2f)" % (name, best_f1, best_thr)
    ax.plot(thresholds, f1_vals, label=label, color=COLORS[name], linewidth=2)
    ax.scatter(best_thr, best_f1, color=COLORS[name],
               s=80, zorder=5, edgecolors="white", linewidths=1.5)

ax.set_xlabel("Classification Threshold", fontsize=12)
ax.set_ylabel("F1-Score", fontsize=12)
ax.set_title("F1-Score Across Thresholds", fontsize=15, fontweight="bold")
ax.legend(loc="lower center", fontsize=10)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

This plot reveals that the default 0.5 threshold is not optimal for any of the models. Each model peaks at a different threshold — the dots mark where each achieves its maximum F1. Random Forest's peak is the highest, confirming its superiority on this metric.

## 7. Confusion Matrices (at best-F1 threshold)

Rather than using the default 0.5 cutoff, we apply each model's **optimal threshold** (the one that maximises F1) and display the resulting confusion matrices.

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

for ax, (name, m) in zip(axes, results.items()):
    best_thr, best_f1 = best_thresholds[name]
    y_pred_best = (m["y_proba"] >= best_thr).astype(int)
    cm = confusion_matrix(y_test, y_pred_best)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Ham", "Spam"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(
        "%s\nThreshold = %.2f  |  F1 = %.4f" % (name, best_thr, best_f1),
        fontsize=12, fontweight="bold",
    )

fig.suptitle(
    "Confusion Matrices (at best-F1 threshold per model)",
    fontsize=15, fontweight="bold", y=1.03,
)
fig.tight_layout()
plt.show()

With optimised thresholds, the confusion matrices tell a clear story: Random Forest achieves the best balance of true positives (spam caught) and false positives (ham misclassified).

## 8. Per-model classification reports

In [ ]:
for name, m in results.items():
    best_thr, _ = best_thresholds[name]
    y_pred_best = (m["y_proba"] >= best_thr).astype(int)
    print("=" * 55)
    print("%s  (threshold = %.2f)" % (name, best_thr))
    print("=" * 55)
    print(classification_report(y_test, y_pred_best, target_names=["ham", "spam"]))
    print()

## 9. Analysis — Which model is best and why?

### Random Forest wins

Random Forest achieves the best scores on almost every metric that matters for imbalanced classification:

| Metric    | Logistic Regression | Random Forest | XGBoost |
|-----------|--------------------:|--------------:|--------:|
| PR-AUC    | 0.9645              | **0.9627**    | 0.9076  |
| F1-Score  | 0.8359              | **0.9164**    | 0.8467  |
| Precision | 1.0000              | **1.0000**    | 0.9280  |
| Recall    | 0.7181              | **0.8456**    | 0.7785  |
| Accuracy  | 0.9623              | **0.9794**    | 0.9623  |

Random Forest dominates because it achieves the best trade-off: it never misclassifies a ham message as spam (perfect precision), while simultaneously catching the most spam (highest recall). That combination is exactly what PR-AUC rewards.

### Why not XGBoost?

XGBoost underperforms despite being a more complex model. Its precision drops to 0.928 — it produces false positives that the other two models avoid entirely. Its recall is lower than Random Forest's, and its PR-AUC of 0.9076 is the worst of the three.

Gradient boosting doesn't always outperform simpler ensembles. On smaller, well-structured TF-IDF feature spaces, Random Forest's bagging approach can generalise better than boosting, which is more prone to overfitting on limited data.

### Why not Logistic Regression?

Logistic Regression matches Random Forest on precision (perfect 1.0) but falls behind significantly on recall (0.7181 vs 0.8456). It misses roughly 28% of spam. As a linear model, it cannot capture non-linear interactions between TF-IDF features that the tree-based ensembles leverage. That said, it remains the simplest and fastest option, and its perfect precision makes it a reasonable choice if minimising false positives is the *only* priority.

### Summary

| Criterion            | Winner             | Reason                                                 |
|----------------------|--------------------|--------------------------------------------------------|
| Best PR-AUC          | Random Forest      | Best precision-recall trade-off across all thresholds  |
| Best F1-Score        | Random Forest      | Highest harmonic mean of precision and recall          |
| Best Precision       | LR / Random Forest | Both achieve 1.0 — zero false positives                |
| Best Recall          | Random Forest      | Catches the most spam (84.6%)                          |
| Fastest / Simplest   | Logistic Regression| Linear model, trains in milliseconds                   |
| **Best Overall**     | **Random Forest**  | Top PR-AUC + top F1 + perfect precision + best recall  |

## 10. Quick inference demo

In [ ]:
from preprocess import clean_text

vectorizer = joblib.load(os.path.join(MODELS_DIR, "tfidf_vectorizer.joblib"))
best_model = models["RandomForest"]

test_messages = [
    "Congratulations! You've won a free iPhone. Click here to claim.",
    "Hey, are we still meeting at 5 today?",
    "URGENT: Your account has been compromised. Verify now.",
    "Can you pick up some milk on the way home?",
]

cleaned = [clean_text(msg) for msg in test_messages]
X_new = vectorizer.transform(cleaned)
probas = best_model.predict_proba(X_new)[:, 1]

print("%-65s  %6s  %s" % ("Message", "P(spam)", "Label"))
print("-" * 90)
for msg, p in zip(test_messages, probas):
    label = "SPAM" if p >= 0.5 else "HAM"
    print("%-65s  %.4f  %s" % (msg[:65], p, label))